# Section 4: Ephemeral Software and Micro-Vulnerabilities

**ITESO | Vulnerability Management and the AI Crossroads**

This notebook demonstrates a concrete problem with AI-generated code: **real vulnerability patterns that static analysis tools can detect, but that CVE-based scanners will never see.**

We'll use `bandit` — a real Python security linter used in production CI/CD pipelines — to scan code samples that exhibit the vulnerability classes most commonly found in AI-generated code (per Pearce et al. 2022 and Perry et al. 2023).

**The core point:** Traditional vulnerability management works by matching library versions against a CVE database. AI-generated code has no library version and no CVE entry. The vulnerability is in the logic, not the dependency — and your scanner will never flag it.

> **Reference:** Pearce et al. 2022, "Asleep at the Keyboard?" — ~40% of Copilot-generated code in security-relevant scenarios contained vulnerabilities. arxiv.org/abs/2108.09293

In [ ]:
import subprocess
import json
import requests
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11

OUTPUT = Path('output')
OUTPUT.mkdir(exist_ok=True)
SAMPLES_DIR = OUTPUT / 'code_samples'
SAMPLES_DIR.mkdir(exist_ok=True)

EPSS_URL = 'https://api.first.org/data/v1/epss'

try:
    result = subprocess.run(['bandit', '--version'], capture_output=True, text=True)
    print(f'bandit: {result.stdout.strip()}')
except FileNotFoundError:
    print('bandit not found — install with: pip install bandit')

print('Setup complete.')

## 1. Vulnerability Patterns in AI-Generated Code

Each sample below represents a real vulnerability class identified in AI code generation research. Each also shows the *fixed* version so students can see the difference.

The key observation: **these look correct at a glance.** They pass syntax checks, they run, and they often pass code review — because the flaw is behavioral, not structural.

In [ ]:
SAMPLES = {
    'sql_injection.py': {
        'description': 'SQL Injection — f-string in query construction',
        'cwe': 'CWE-89',
        'code': '''\
import sqlite3

def get_user_orders(username: str, db_path: str = "orders.db"):
    """Retrieve all orders for a given username."""
    conn = sqlite3.connect(db_path)
    # AI-generated: f-string directly in query — SQL injection vector
    query = f"SELECT * FROM orders WHERE username = '{username}'"
    return conn.execute(query).fetchall()

def safe_get_user_orders(username: str, db_path: str = "orders.db"):
    """Fixed: parameterised query."""
    conn = sqlite3.connect(db_path)
    return conn.execute("SELECT * FROM orders WHERE username = ?", (username,)).fetchall()
''',
    },
    'path_traversal.py': {
        'description': 'Path Traversal — unsanitised user input in file path',
        'cwe': 'CWE-22',
        'code': '''\
import os

UPLOAD_DIR = "/var/app/uploads"

def read_user_file(filename: str) -> str:
    """Return contents of a user-uploaded file."""
    # AI-generated: no path sanitisation — allows ../../etc/passwd
    file_path = os.path.join(UPLOAD_DIR, filename)
    with open(file_path) as f:
        return f.read()

def safe_read_user_file(filename: str) -> str:
    """Fixed: realpath check enforces containment within UPLOAD_DIR."""
    safe_path = os.path.realpath(os.path.join(UPLOAD_DIR, filename))
    if not safe_path.startswith(os.path.realpath(UPLOAD_DIR) + os.sep):
        raise ValueError("Access denied: path traversal attempt")
    with open(safe_path) as f:
        return f.read()
''',
    },
    'command_injection.py': {
        'description': 'Command Injection — shell=True with user-controlled input',
        'cwe': 'CWE-78',
        'code': '''\
import subprocess

def ping_host(hostname: str) -> str:
    """Ping a host and return output."""
    # AI-generated: shell=True + user string = command injection
    result = subprocess.run(
        f"ping -c 4 {hostname}", shell=True, capture_output=True, text=True
    )
    return result.stdout

def safe_ping_host(hostname: str) -> str:
    """Fixed: argument list, no shell, hostname validated."""
    if not hostname.replace(".", "").replace("-", "").isalnum():
        raise ValueError("Invalid hostname")
    result = subprocess.run(["ping", "-c", "4", hostname], capture_output=True, text=True)
    return result.stdout
''',
    },
    'weak_crypto.py': {
        'description': 'Weak Cryptography — MD5 for password hashing',
        'cwe': 'CWE-327',
        'code': '''\
import hashlib, secrets

def hash_password_weak(password: str) -> str:
    """Hash a password for storage."""
    # AI-generated: MD5 is cryptographically broken for passwords
    return hashlib.md5(password.encode()).hexdigest()

def hash_password_safe(password: str) -> str:
    """Fixed: scrypt is memory-hard and appropriate for password storage."""
    salt = secrets.token_hex(16)
    dk = hashlib.scrypt(password.encode(), salt=salt.encode(), n=2**14, r=8, p=1)
    return f"{salt}${dk.hex()}"
''',
    },
}

for filename, meta in SAMPLES.items():
    (SAMPLES_DIR / filename).write_text(meta['code'])

print(f'{len(SAMPLES)} code samples written to {SAMPLES_DIR}')

## 2. Running Bandit — Static Security Analysis

Bandit is a real tool. It's used in production CI/CD pipelines at companies like Netflix, Dropbox, and across the Python open-source ecosystem. Running it here shows what a security scan of AI-generated code would actually look like.

In [ ]:
def run_bandit(path: Path) -> dict:
    result = subprocess.run(
        ['bandit', '-r', str(path), '-f', 'json', '-q'],
        capture_output=True, text=True
    )
    try:
        return json.loads(result.stdout)
    except json.JSONDecodeError:
        return {'results': [], 'error': result.stderr}


all_findings = []
print('Bandit scan results:\n')

for filename, meta in SAMPLES.items():
    report   = run_bandit(SAMPLES_DIR / filename)
    findings = report.get('results', [])
    print(f'  {filename}  —  {meta["description"]}')
    if findings:
        for f in findings:
            print(f'    [{f["issue_severity"]}/{f["issue_confidence"]}] '
                  f'Line {f["line_number"]}: {f["issue_text"]}')
            all_findings.append({'file': filename, 'cwe': meta['cwe'],
                                  'severity': f['issue_severity'],
                                  'issue': f['issue_text']})
    else:
        print('    (no findings — is bandit installed?)')
    print()

if all_findings:
    df = pd.DataFrame(all_findings)
    print(f'Total findings across {len(SAMPLES)} files: {len(df)}')
    print(df['severity'].value_counts().rename('count').to_string())

## 3. Why CVE Scanners Can't Help Here

The same vulnerability classes above map to real, high-EPSS CVEs in third-party libraries. The difference: when the flaw is in a known library, there's a CVE, an EPSS score, a patch, and a scanner alert. When it's in AI-generated code, there's nothing.

In [ ]:
# Real CVEs in the same vulnerability classes as our samples
REFERENCE_CVES = {
    'SQL Injection (CWE-89)':     ['CVE-2023-20887', 'CVE-2022-1388'],
    'Path Traversal (CWE-22)':    ['CVE-2021-41773', 'CVE-2022-22965'],
    'Command Injection (CWE-78)': ['CVE-2021-44228', 'CVE-2023-46604'],
    'Weak Crypto (CWE-327)':      ['CVE-2022-0778',  'CVE-2023-0215'],
}

all_cves = [c for cves in REFERENCE_CVES.values() for c in cves]
try:
    r = requests.get(EPSS_URL, params={'cve': ','.join(all_cves)}, timeout=15)
    r.raise_for_status()
    epss_lookup = {d['cve']: float(d['epss']) * 100 for d in r.json().get('data', [])}
except Exception as e:
    print(f'EPSS API unavailable ({e})')
    epss_lookup = {}

print('Vulnerability class          CVE                   EPSS    Scanner alert?  AI-gen alert?')
print('-' * 90)
for vclass, cves in REFERENCE_CVES.items():
    for cve in cves:
        score = epss_lookup.get(cve)
        score_str = f'{score:.2f}%' if score is not None else 'N/A   '
        short_class = vclass.split('(')[0].strip()[:28]
        print(f'{short_class:<30} {cve:<22} {score_str:<8} YES             NO')

print()
print('→ The same flaw in AI-generated code has no CVE, no EPSS score, and no scanner alert.')
print('  It is identical logic. The only difference is provenance.')

## 4. The Risk Surface: What Your Scanner Sees vs. Reality

Visualising coverage: in a traditional codebase, CVE scanners cover (nearly) everything. In a mixed AI-augmented codebase, a growing portion is invisible to those scanners.

In [ ]:
def make_inventory(n_libs, n_reviewed, n_unreviewed, seed=0):
    rng = np.random.default_rng(seed)
    items = []
    for _ in range(n_libs):
        items.append(('lib',         rng.uniform(0.05, 0.95), rng.uniform(0.05, 0.95),
                       int(rng.integers(60, 280)), '#4a90d9'))
    for _ in range(n_reviewed):
        items.append(('ai_reviewed', rng.uniform(0.05, 0.95), rng.uniform(0.05, 0.95),
                       int(rng.integers(30, 140)), '#60b26e'))
    for _ in range(n_unreviewed):
        items.append(('ai_blind',    rng.uniform(0.05, 0.95), rng.uniform(0.05, 0.95),
                       int(rng.integers(25, 110)), '#e06c5a'))
    return items


fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
fig.subplots_adjust(top=0.88, bottom=0.14)

scenarios = [
    (make_inventory(18, 0, 0, 0),  'Traditional Codebase\n(18 known libs — CVE-trackable)'),
    (make_inventory(10, 4, 6, 10), 'AI-Augmented Codebase\n(10 libs + 10 AI-generated components)'),
]

for ax, (inv, title) in zip(axes, scenarios):
    for kind, x, y, size, color in inv:
        ax.scatter(x, y, s=size, color=color, alpha=0.72, edgecolors='white', lw=0.6)
    n_visible = sum(1 for c in inv if c[0] == 'lib')
    coverage  = n_visible / len(inv) * 100
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_facecolor('#f8f9fa')
    ax.set_title(title, fontweight='bold', fontsize=10)
    ax.text(0.5, -0.06, f'CVE scanner coverage: {coverage:.0f}% of components',
            ha='center', transform=ax.transAxes, fontsize=9, color='#555')

legend_handles = [
    mpatches.Patch(color='#4a90d9', label='Third-party library (CVE-trackable)'),
    mpatches.Patch(color='#60b26e', label='AI-generated, human-reviewed'),
    mpatches.Patch(color='#e06c5a', label='AI-generated, no review (invisible to CVE scanners)'),
]
fig.legend(handles=legend_handles, loc='lower center', ncol=3,
           fontsize=9, bbox_to_anchor=(0.5, -0.03))
fig.suptitle('Software Risk Surface: What Your CVE Scanner Sees vs. Reality',
             fontsize=13, fontweight='bold')
fig.text(0.5, 0.01,
         'The red components exist, run in production, and may contain real vulnerabilities — '
         'but no CVE will ever be filed against them.',
         ha='center', fontsize=8, color='#666', style='italic', transform=fig.transFigure)

plt.savefig(OUTPUT / '04_risk_surface.png', bbox_inches='tight')
plt.show()
print('Chart saved.')

## 5. So What Do You Do About It?

There is no mature, universal answer. The current practical options are:

| Approach | What it catches | Limitation |
|----------|----------------|------------|
| **Static analysis (bandit, semgrep)** | Known bad patterns (injection, weak crypto) | Misses logic-level flaws; high false-positive rate |
| **Mandatory human review for AI-generated code** | Anything a reviewer can spot | Expensive; review fatigue; scales poorly |
| **Runtime monitoring (eBPF, EDR)** | Anomalous behavior in production | Reactive, not preventive; complex to deploy |
| **Restrict AI codegen to non-sensitive paths** | Prevents the problem in security-critical code | Hard to enforce; definition of "sensitive" is fuzzy |

**The honest answer for students:** The tooling gap is real, the problem is growing, and the industry hasn't solved it yet. This is where the security engineering work of the next few years will happen.

## 6. Try It Yourself

**Exercise:** Ask an AI assistant (ChatGPT, Claude, Copilot) to write you a Python function that reads a user-specified file from a known directory. Save the output to `output/code_samples/student_test.py` and run bandit against it:

```bash
bandit output/code_samples/student_test.py -v
```

Did the AI generate safe code? Did it vary when you asked differently?